# RUKOPYS Qwen3-VL Stage 2B: Build Hard-type Augmented Dataset

This notebook prepares the crop-level hard-type training data for Stage 2B. It materializes augmented crop images and writes a JSONL manifest so the fine-tune notebook can run faster and repeat training without rebuilding augmentation.

Output artifacts:
- `stage2b_hardtype_aug_samples.jsonl`
- `gold_validation_records.jsonl`
- `prompt_config.json`
- `crops/*.jpg`


In [1]:
# Kaggle dependency cell. No GPU is required for this build notebook.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "pandas==2.2.2",
            "pillow<12",
            "datasets",
        ],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)


In [3]:
%pip install pillow

   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.1 MB 6.0 MB/s eta 0:00:02
   ---------- ----------------------------- 1.8/7.1 MB 4.9 MB/s eta 0:00:02
   ---------------- ----------------------- 2.9/7.1 MB 4.4 MB/s eta 0:00:01
   ------------------- -------------------- 3.4/7.1 MB 4.0 MB/s eta 0:00:01
   -------------------------- ------------- 4.7/7.1 MB 4.5 MB/s eta 0:00:01
   -------------------------------------- - 6.8/7.1 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 7.1/7.1 MB 5.4 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
import hashlib
import json
import random
import re
from collections import defaultdict
from pathlib import Path

from PIL import Image, ImageEnhance, ImageFilter

Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED)

DATASET_ROOT = Path(r"C:\Users\HP\source\rukopys_data")
OUTPUT_DIR = Path(r"C:\Users\HP\source\Handwritten_to_Data_Ukraine\artifacts\stage2b_hardtype_aug_data")
CROP_DIR = OUTPUT_DIR / "crops"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.12
GOLD_CROP_LIMIT = None
MIN_BOX_SIDE = 8
CROP_PAD_RATIO = 0.0
JPEG_QUALITY = 95

HARD_TYPES = {"formula", "table", "annotation"}
REPLAY_TYPES = {"handwritten", "printed"}
HARD_AUG_COPIES_BY_TYPE = {
    "formula": 3,
    "annotation": 3,
    "table": 7,
}
REPLAY_PER_HARD = 2.0
MIN_REPLAY_SAMPLES = 2000
MAX_REPLAY_SAMPLES = 12000
REPLAY_KEEP_RATIO_BY_TYPE = {
    "handwritten": 0.5,
    "printed": 1.0,
}
AUG_ROTATE_DEGREES = {
    "formula": 1.2,
    "table": 0.8,
    "annotation": 2.0,
    "handwritten": 1.0,
    "printed": 0.8,
}

PROMPT_VERSION = "hybrid_prompt_v2"

SOURCE_HINTS = {
    "dictation": "Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.",
    "archive": "Historical Ukrainian/Cyrillic document. Preserve old spelling; do not modernize.",
    "school": "School homework. It may contain corrections, teacher marks, formulas, and mixed handwriting/print.",
    "university": "University exam/coursework. It may contain formulas, tables, chemistry notation, and technical symbols.",
}
DEFAULT_SOURCE_HINT = "Read only visible characters from this crop."

SPECIAL_TEXT_MARKER_RULES = (
    "Use [illegible] only for unreadable words inside an otherwise legible text region. "
    "Use ~~word~~ for visible strikethrough and ~~old~~{new} for visible correction."
)

STAGE_B_GUARDRAILS = (
    "The final transcription must be supported by the crop. "
    "Do not complete missing words from source hint, language prior, or canonical dictation text. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text. No JSON, no Markdown, no explanation."
)

CROP_PROMPTS = {
    "handwritten": (
        "Transcribe the visible handwritten text exactly. Preserve punctuation, line content, "
        "corrections, spelling mistakes, capitalization, digits, abbreviations, quotes, hyphens, "
        "line-final dashes, visible spacing, and strikethrough markers. Return only text."
    ),
    "printed": (
        "Transcribe the visible printed or typed text exactly. Preserve punctuation, line content, "
        "corrections, spelling mistakes, capitalization, digits, abbreviations, quotes, hyphens, "
        "line-final dashes, visible spacing, and strikethrough markers. Return only text."
    ),
    "annotation": "Read this short annotation or teacher mark. Return only the exact visible text.",
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Preserve visible "
        "symbols, indices, superscripts, subscripts, arrows, fractions, matrix/determinant structure, punctuation, "
        "numbering, and strikethrough/correction markers. Do not solve, simplify, normalize, explain, or convert "
        "old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, and visible "
        "spelling mistakes. Do not infer missing cells, do not rebalance columns, do not summarize, and do not explain."
    ),
    "image": "Return an empty string.",
    "graph": "Return an empty string.",
    "default": "Transcribe the visible content exactly. Preserve punctuation, corrections, and visible spacing. Return only text.",
}

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
SCORABLE_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]


In [5]:

def get_dataset_root():
    root = Path(DATASET_ROOT)
    train_meta = root / "train" / "metadata.jsonl"
    if not train_meta.exists():
        raise FileNotFoundError(f"Expected train/metadata.jsonl under {root}")
    return root


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def resolve_image_path(root, split, file_name):
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [root / split / file_name, root / file_name]
    for candidate_name in candidate_names:
        candidates.extend([root / split / "images" / candidate_name, root / split / candidate_name])
    for p in candidates:
        if p.exists():
            return str(p)
    return str(root / split / "images" / candidate_names[0])


def clamp_box(box, w, h):
    if not isinstance(box, list) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(w, x1)), max(0, min(w, x2))))
    y1, y2 = sorted((max(0, min(h, y1)), max(0, min(h, y2))))
    if x2 - x1 < MIN_BOX_SIDE or y2 - y1 < MIN_BOX_SIDE:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    return value if value in VALID_TYPES else "handwritten"


def normalize_source(value):
    value = str(value or "").strip().lower()
    return value if value in SOURCE_HINTS else "default"


def get_source_hint(source):
    return SOURCE_HINTS.get(normalize_source(source), DEFAULT_SOURCE_HINT)


def build_crop_prompt(rtype, source=None):
    rtype = normalize_type(rtype)
    type_prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    if rtype in {"image", "graph"}:
        return type_prompt
    return "\n".join([get_source_hint(source), type_prompt, SPECIAL_TEXT_MARKER_RULES, STAGE_B_GUARDRAILS])


def text_is_sane(text):
    if text is None:
        return False
    text = str(text).strip()
    if not text or len(text) > 260:
        return False
    if re.search(r"(.)\1{10,}", text):
        return False
    bad = sum(ch in "\ufffdï¿½" for ch in text)
    return bad == 0


def region_is_crop_candidate(region):
    rtype = normalize_type(region.get("type"))
    if rtype not in SCORABLE_TYPES:
        return False
    if str(region.get("language", "uk")).lower() == "other":
        return False
    if str(region.get("legibility", "legible")).lower() == "illegible":
        return False
    return text_is_sane(region.get("text"))


def stratified_split(records, val_ratio=0.12):
    by_source = defaultdict(list)
    for row in records:
        by_source[row.get("source", "unknown")].append(row)
    train_rows, val_rows = [], []
    rng = random.Random(SEED)
    for _, items in by_source.items():
        items = list(items)
        rng.shuffle(items)
        if len(items) <= 1:
            n_val = 0
        else:
            n_val = min(max(1, int(round(len(items) * val_ratio))), len(items) - 1)
        val_rows.extend(items[:n_val])
        train_rows.extend(items[n_val:])
    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    return train_rows, val_rows


In [6]:

def make_crop_sample(record, region, root, split):
    w = max(1, int(record.get("image_width") or 1))
    h = max(1, int(record.get("image_height") or 1))
    box = clamp_box(region.get("bbox"), w, h)
    if box is None or not region_is_crop_candidate(region):
        return None
    rtype = normalize_type(region.get("type"))
    return {
        "task": "crop_ocr",
        "source_image_path": resolve_image_path(root, split, record["file_name"]),
        "source_file_name": record["file_name"],
        "bbox": box,
        "region_type": rtype,
        "prompt": build_crop_prompt(rtype, source=record.get("source")),
        "answer": str(region.get("text") or ""),
        "source": record.get("source", "unknown"),
        "augment": False,
        "aug_id": 0,
    }


def balanced_take(samples, limit):
    if limit is None or len(samples) <= limit:
        return samples
    buckets = defaultdict(list)
    for s in samples:
        key = (s.get("source", "unknown"), s.get("region_type", s.get("task")))
        buckets[key].append(s)
    rng = random.Random(SEED)
    for items in buckets.values():
        rng.shuffle(items)
    selected = []
    keys = list(buckets.keys())
    while len(selected) < limit and keys:
        next_keys = []
        for key in keys:
            if buckets[key] and len(selected) < limit:
                selected.append(buckets[key].pop())
            if buckets[key]:
                next_keys.append(key)
        keys = next_keys
    rng.shuffle(selected)
    return selected


def apply_replay_keep_ratio(replay):
    by_type = defaultdict(list)
    for sample in replay:
        by_type[sample.get("region_type")].append(sample)

    kept = []
    for rtype, items in sorted(by_type.items()):
        keep_ratio = REPLAY_KEEP_RATIO_BY_TYPE.get(rtype, 1.0)
        keep_n = int(round(len(items) * keep_ratio))
        kept.extend(items[:keep_n])
    return kept


def make_stage2b_crop_mix(crop_samples):
    rng = random.Random(SEED)
    hard = [dict(s, augment=False, aug_id=0) for s in crop_samples if s.get("region_type") in HARD_TYPES]
    replay_pool = [dict(s, augment=False, aug_id=0) for s in crop_samples if s.get("region_type") in REPLAY_TYPES]

    replay_limit = int(max(MIN_REPLAY_SAMPLES, min(MAX_REPLAY_SAMPLES, len(hard) * REPLAY_PER_HARD)))
    replay_before_ratio = balanced_take(replay_pool, replay_limit)
    replay = apply_replay_keep_ratio(replay_before_ratio)
    rng.shuffle(replay)

    augmented = []
    for s in hard:
        augmented.append(s)
        aug_copies = HARD_AUG_COPIES_BY_TYPE.get(s.get("region_type"), 0)
        for aug_id in range(1, aug_copies + 1):
            t = dict(s)
            t["augment"] = True
            t["aug_id"] = aug_id
            augmented.append(t)

    mixed = augmented + replay
    rng.shuffle(mixed)

    type_counts = defaultdict(int)
    aug_counts = defaultdict(int)
    replay_counts = defaultdict(int)
    for s in mixed:
        type_counts[s.get("region_type")] += 1
        if s.get("augment"):
            aug_counts[s.get("region_type")] += 1
        if s.get("region_type") in REPLAY_TYPES:
            replay_counts[s.get("region_type")] += 1
    print("Stage2B crop mix type counts:", dict(sorted(type_counts.items())))
    print("Stage2B augmented counts:", dict(sorted(aug_counts.items())))
    print("Stage2B replay counts after keep ratio:", dict(sorted(replay_counts.items())))
    return mixed


def build_samples(records, root, split, crop_limit=None):
    rng = random.Random(SEED)
    rows = list(records)
    rng.shuffle(rows)
    crop_samples = []
    for record in rows:
        for region in record.get("regions") or []:
            crop = make_crop_sample(record, region, root, split)
            if crop is not None:
                crop_samples.append(crop)
    crop_samples = balanced_take(crop_samples, crop_limit)
    crop_samples = make_stage2b_crop_mix(crop_samples)
    print(f"{split}: crop={len(crop_samples)}")
    return crop_samples


In [7]:

def stable_aug_rng(sample):
    key = f"{sample.get('source_image_path')}|{sample.get('bbox')}|{sample.get('region_type')}|{sample.get('aug_id', 0)}|{SEED}"
    return random.Random(key)


def apply_light_ocr_augment(img, region_type, rng):
    deg = AUG_ROTATE_DEGREES.get(region_type, 1.0)
    angle = rng.uniform(-deg, deg)
    if abs(angle) > 0.05:
        img = img.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True, fillcolor=(255, 255, 255))

    if rng.random() < 0.90:
        img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.90, 1.10))
    if rng.random() < 0.90:
        img = ImageEnhance.Contrast(img).enhance(rng.uniform(0.90, 1.16))
    if rng.random() < 0.35:
        img = ImageEnhance.Sharpness(img).enhance(rng.uniform(0.85, 1.25))

    blur_prob = 0.10 if region_type in {"formula", "table"} else 0.16
    if rng.random() < blur_prob:
        img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.15, 0.45)))
    return img


def crop_with_padding_and_optional_aug(sample, pad_ratio=CROP_PAD_RATIO):
    rng = stable_aug_rng(sample)
    local_pad_ratio = pad_ratio
    if sample.get("augment"):
        local_pad_ratio = pad_ratio * rng.uniform(0.7, 2.0)

    with Image.open(sample["source_image_path"]) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = sample["bbox"]
        pad = int(round(max(x2 - x1, y2 - y1) * local_pad_ratio))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        crop = img.crop((x1, y1, x2, y2))

    if sample.get("augment"):
        crop = apply_light_ocr_augment(crop, sample.get("region_type", "default"), rng)
    return crop


def sample_cache_name(sample, idx):
    key = json.dumps({
        "file": sample.get("source_file_name"),
        "bbox": sample.get("bbox"),
        "type": sample.get("region_type"),
        "aug_id": sample.get("aug_id", 0),
        "seed": SEED,
    }, sort_keys=True, ensure_ascii=False)
    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()[:12]
    rtype = re.sub(r"[^a-z0-9_-]+", "_", sample.get("region_type", "crop"))
    return f"{idx:06d}_{rtype}_aug{sample.get('aug_id', 0)}_{digest}.jpg"


def materialize_crops(samples):
    manifest = []
    for idx, sample in enumerate(samples):
        rel_path = Path("crops") / sample_cache_name(sample, idx)
        out_path = OUTPUT_DIR / rel_path
        if not out_path.exists():
            crop = crop_with_padding_and_optional_aug(sample)
            crop.save(out_path, quality=JPEG_QUALITY, optimize=True)
        row = dict(sample)
        row["image_path"] = str(rel_path).replace("\\", "/")
        row["relative_image_path"] = row["image_path"]
        row.pop("source_image_path", None)
        manifest.append(row)
        if (idx + 1) % 1000 == 0:
            print(f"materialized {idx + 1}/{len(samples)} crops", flush=True)
    return manifest


In [8]:
root = get_dataset_root()
gold_records = read_jsonl(root / "train" / "metadata.jsonl")
gold_train_records, gold_val_records = stratified_split(gold_records, VAL_RATIO)
print(f"Gold train pages={len(gold_train_records)} val pages={len(gold_val_records)}")

samples = build_samples(gold_train_records, root, "train", crop_limit=GOLD_CROP_LIMIT)
if not samples:
    raise RuntimeError("No Stage 2B crop samples were built. Check dataset metadata and filters.")

manifest = materialize_crops(samples)
samples_path = OUTPUT_DIR / "stage2b_hardtype_aug_samples.jsonl"
val_path = OUTPUT_DIR / "gold_validation_records.jsonl"
prompt_config_path = OUTPUT_DIR / "prompt_config.json"

write_jsonl(samples_path, manifest)
write_jsonl(val_path, gold_val_records)
prompt_config_path.write_text(
    json.dumps({
        "prompt_version": PROMPT_VERSION,
        "stage": "stage2b_hardtype_aug_dataset",
        "hard_types": sorted(list(HARD_TYPES)),
        "hard_aug_copies_by_type": HARD_AUG_COPIES_BY_TYPE,
        "replay_keep_ratio_by_type": REPLAY_KEEP_RATIO_BY_TYPE,
        "source_hints": SOURCE_HINTS,
        "default_source_hint": DEFAULT_SOURCE_HINT,
        "crop_prompts": CROP_PROMPTS,
        "special_text_marker_rules": SPECIAL_TEXT_MARKER_RULES,
        "stage_b_guardrails": STAGE_B_GUARDRAILS,
        "crop_prompt_assembly": "source_hint + type_prompt + special_text_marker_rules + stage_b_guardrails",
        "crop_pad_ratio": CROP_PAD_RATIO,
        "seed": SEED,
        "sample_count": len(manifest),
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Wrote samples:", samples_path, "rows=", len(manifest))
print("Wrote validation records:", val_path, "rows=", len(gold_val_records))
print("Wrote prompt config:", prompt_config_path)
print("Crop dir:", CROP_DIR)


Gold train pages=1171 val pages=159
Stage2B crop mix type counts: {'annotation': 1616, 'formula': 8972, 'handwritten': 2639, 'printed': 246, 'table': 920}
Stage2B augmented counts: {'annotation': 1212, 'formula': 6729, 'table': 805}
Stage2B replay counts after keep ratio: {'handwritten': 2639, 'printed': 246}
train: crop=14393
materialized 1000/14393 crops
materialized 2000/14393 crops
materialized 3000/14393 crops
materialized 4000/14393 crops
materialized 5000/14393 crops
materialized 6000/14393 crops
materialized 7000/14393 crops
materialized 8000/14393 crops
materialized 9000/14393 crops
materialized 10000/14393 crops
materialized 11000/14393 crops
materialized 12000/14393 crops
materialized 13000/14393 crops
materialized 14000/14393 crops
Wrote samples: C:\Users\HP\source\Handwritten_to_Data_Ukraine\artifacts\stage2b_hardtype_aug_data\stage2b_hardtype_aug_samples.jsonl rows= 14393
Wrote validation records: C:\Users\HP\source\Handwritten_to_Data_Ukraine\artifacts\stage2b_hardtype_a

In [ ]:
# Optional smoke check: open the first cached crop and print its training prompt.
RUN_SMOKE_CHECK = True

if RUN_SMOKE_CHECK and manifest:
    first = manifest[0]
    print(json.dumps({k: first[k] for k in ["image_path", "region_type", "source", "augment", "aug_id", "answer"]}, ensure_ascii=False, indent=2))
    print("Prompt preview:\n", first["prompt"][:1200])
    display(Image.open(OUTPUT_DIR / first["image_path"]))
